In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent)
print(path)
sys.path.insert(1, path)

import numpy as np
import pandas as pd
import skforecast

print(skforecast.__version__)

c:\Users\jaesc2\GitHub\skforecast
0.20.0


In [ ]:
import time
import numpy as np
from sklearn.datasets import make_regression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

def benchmark_paths():
    # 1. Setup: Create dummy data and train the models
    print("Generating data and training models...")
    X, y = make_regression(n_samples=1000, n_features=15, random_state=42)
    steps = 500
    
    dt = DecisionTreeRegressor(max_depth=10, random_state=42)
    dt.fit(X, y)
    
    rf = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)
    rf.fit(X, y)

    print(f"Simulating recursive loop for {steps} steps...\n")
    print("-" * 50)

    # ==========================================
    # DECISION TREE BENCHMARK
    # ==========================================
    print("MODEL: DecisionTreeRegressor")
    X_test = X[0].reshape(1, -1)
    # Normal Path
    start_time = time.perf_counter()
    preds_normal_dt = np.zeros(steps)
    for i in range(steps):
        # The standard scikit-learn way
        preds_normal_dt[i] = dt.predict(X_test).item()
    dt_normal_time = time.perf_counter() - start_time
    
    # Fast Path
    start_time = time.perf_counter()
    preds_fast_dt = np.zeros(steps)
    for i in range(steps):
        # The Cython fast path
        X_f32 = np.ascontiguousarray(X_test, dtype=np.float32)
        preds_fast_dt[i] = dt.tree_.predict(X_f32).item()
    dt_fast_time = time.perf_counter() - start_time

    # Verification
    dt_match = np.allclose(preds_normal_dt, preds_fast_dt)
    
    print(f"Normal Path Time: {dt_normal_time:.4f} seconds")
    print(f"Fast Path Time:   {dt_fast_time:.4f} seconds")
    print(f"Speedup:          {dt_normal_time / dt_fast_time:.1f}x faster")
    print(f"Predictions Match: {dt_match}\n")
    print("-" * 50)

    # ==========================================
    # RANDOM FOREST BENCHMARK
    # ==========================================
    print("MODEL: RandomForestRegressor (50 Trees)")
    X_test = X[0].reshape(1, -1)
    # Normal Path
    start_time = time.perf_counter()
    preds_normal_rf = np.zeros(steps)
    for i in range(steps):
        preds_normal_rf[i] = rf.predict(X_test).item()
    rf_normal_time = time.perf_counter() - start_time
    
    # Fast Path
    start_time = time.perf_counter()
    preds_fast_rf = np.zeros(steps)
    estimators = rf.estimators_ # Cache this outside the loop!
    
    for i in range(steps):
        # The Cython fast path mapped across the ensemble
        X_f32 = np.ascontiguousarray(X_test, dtype=np.float32)
        preds_fast_rf[i] = np.mean(
            [tree.tree_.predict(X_f32).item() for tree in estimators]
        )
    rf_fast_time = time.perf_counter() - start_time

    # Verification
    rf_match = np.allclose(preds_normal_rf, preds_fast_rf)

    print(f"Normal Path Time: {rf_normal_time:.4f} seconds")
    print(f"Fast Path Time:   {rf_fast_time:.4f} seconds")
    print(f"Speedup:          {rf_normal_time / rf_fast_time:.1f}x faster")
    print(f"Predictions Match: {rf_match}\n")
    print("-" * 50)


benchmark_paths()